# kinematics: SO-101 forward and inverse kinematics

Library notebook for `gambit_arm`. Forward kinematics (FK) of the SO-ARM101 built directly
from the joint origins and axes in `gambit_description/urdf/so101.urdf.xacro`, a numeric
Jacobian, and a damped-least-squares (DLS) inverse kinematics (IK) for "put the gripper tip
here, approaching from this direction".

Milestone: *Hardware and execution* (pick-and-place needs joint targets for each square).
The C++ mirror is `gambit_cpp/libs/kinematics` (FK and IK both TODO); its test checks the
same zero-pose numbers as the self-check below.

**TODO**
- [ ] parse the URDF joint chain base_link -> gripper_frame_link (with a hardcoded fallback)
- [ ] FK (4x4 tool pose), numeric Jacobian
- [ ] simple DLS IK for position + approach direction
- [ ] joint-limit handling better than clipping (e.g. weighted DLS or null-space push away
      from limits); seeding strategy (elbow-up branch) so IK never flips configuration
- [ ] top-down orientation constraint: approach within `max_tilt_deg` of vertical, choose the
      tilt per square (far ranks need tilt), and pick wrist_roll so the fingers align with
      the board's files/ranks
- [ ] compare FK against the real arm after calibration (tape measure / repeatability run)
- [ ] precompute and cache IK for all 64 squares at hover and grasp height
- [ ] write the self-checks at the bottom as asserts

In [ ]:
import math
import xml.etree.ElementTree as ET
import numpy as np

%run ./square_map.ipynb

## Joint chain from the URDF

URDF convention: each joint's transform from parent link to child link is

$$T = \mathrm{Trans}(xyz)\;R_z(y)\,R_y(p)\,R_x(r)\;\mathrm{Rot}(\text{axis}, q)$$

We follow the chain `base_link -> shoulder_link -> upper_arm_link -> lower_arm_link ->
wrist_link -> gripper_link -> gripper_frame_link`. The last joint (`gripper_frame_joint`) is
fixed and defines the **tool frame**: its z axis is the approach direction (pointing out of
the fingers), its origin is the point between the fingertips we call "the gripper tip".

`HARDCODED_CHAIN` holds the same numbers copied from the xacro (Nov 2026), used if parsing
fails (e.g. the description package is missing in Colab).

In [ ]:
ARM_JOINTS = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll"]
TOOL_LINK = "gripper_frame_link"

# (name, type, xyz, rpy, axis, lower, upper) from gambit_description/urdf/so101.urdf.xacro
HARDCODED_CHAIN = [
    ("shoulder_pan",  "revolute", (0.0388353, -8.97657e-09, 0.0624), (3.14159, 4.18253e-17, -3.14159), (0, 0, 1), -1.91986, 1.91986),
    ("shoulder_lift", "revolute", (-0.0303992, -0.0182778, -0.0542), (-1.5708, -1.5708, 0), (0, 0, 1), -1.74533, 1.74533),
    ("elbow_flex",    "revolute", (-0.11257, -0.028, 1.73763e-16), (-3.63608e-16, 8.74301e-16, 1.5708), (0, 0, 1), -1.69, 1.69),
    ("wrist_flex",    "revolute", (-0.1349, 0.0052, 3.62355e-17), (4.02456e-15, 8.67362e-16, -1.5708), (0, 0, 1), -1.65806, 1.65806),
    ("wrist_roll",    "revolute", (5.55112e-17, -0.0611, 0.0181), (1.5708, 0.0486795, 3.14159), (0, 0, 1), -2.74385, 2.84121),
    ("gripper_frame_joint", "fixed", (-0.0079, -0.000218121, -0.0981274), (0, 3.14159, 0), (0, 0, 0), 0.0, 0.0),
]

URDF_PATH = REPO_ROOT / "gambit_description" / "urdf" / "so101.urdf.xacro"

def parse_chain(path=URDF_PATH, base="base_link", tip=TOOL_LINK) -> list:
    """Joint chain base -> tip as a list of (name, type, xyz, rpy, axis, lower, upper).
    Only <joint> elements with a type attribute are kinematic (the ros2_control block reuses
    the <joint> tag without one)."""
    # Design:
    # - xml.etree.ElementTree: iterate <joint> elements, keep only kinematic types
    #   (revolute / continuous / prismatic / fixed), index them by child link.
    # - Walk from the tip link up through "which joint has this as child?" until base; reverse.
    #   Walking up is easier than down because each link has exactly one parent joint.
    # - Missing <origin>/<axis>/<limit> default to zeros. Skip joints whose attributes are
    #   xacro expressions like $(arg base_xyz) (float() fails) - world_to_base is one.
    # - Raise if the walk dead-ends: a silently short chain gives a wrong FK.
    # TODO: implement, then compare against HARDCODED_CHAIN (self-check).
    raise NotImplementedError

# Until parse_chain works we use the hardcoded copy. Once it does:
#   try parse_chain(); on failure (file missing in Colab, unparsed xacro) print the error and
#   fall back to HARDCODED_CHAIN; record which one in CHAIN_SOURCE.
CHAIN, CHAIN_SOURCE = HARDCODED_CHAIN, "HARDCODED_CHAIN"

JOINT_LOWER = np.array([c[5] for c in CHAIN if c[1] != "fixed"])
JOINT_UPPER = np.array([c[6] for c in CHAIN if c[1] != "fixed"])
[c[0] for c in CHAIN], CHAIN_SOURCE

## Forward kinematics

Multiply the joint transforms. `fk(q)` returns the 4x4 tool pose in the arm base frame
(= world frame, since `arm.base_xyz` is the origin; add it if the arm moves).

Sign conventions worth knowing (they surprised us): the shoulder_pan origin flips its z axis
(rpy pi, 0, -pi), so **positive pan swings the tool toward -y** (the arm's right). At q = 0
the arm points straight out along +x, horizontal.

In [ ]:
def rpy_matrix(r: float, p: float, y: float) -> np.ndarray:
    """3x3 rotation from URDF roll-pitch-yaw (fixed axes): R = Rz(y) Ry(p) Rx(r)."""
    # Design: build the three elementary rotations and multiply in that order. The order is
    # the classic URDF pitfall; test with a single nonzero angle at a time.
    # TODO: implement
    raise NotImplementedError

def axis_angle_matrix(axis, q: float) -> np.ndarray:
    """Rodrigues rotation about a unit axis."""
    # Design: R = I + sin(q) K + (1 - cos(q)) K^2, K = skew matrix of the normalized axis.
    # TODO: implement
    raise NotImplementedError

def homogeneous(R=np.eye(3), t=(0, 0, 0)) -> np.ndarray:
    """4x4 transform from rotation R and translation t."""
    # TODO: implement (1-3 lines)
    raise NotImplementedError

# Precompute the fixed origin transforms once (each joint's Trans(xyz) Rpy(rpy)), e.g.
#   _ORIGINS = [homogeneous(rpy_matrix(*c[3]), c[2]) for c in CHAIN]
# TODO: enable once rpy_matrix / homogeneous work.
_ORIGINS = None

def fk_all(q, base_xyz=None) -> list:
    """List of 4x4 frames after each joint in CHAIN (last = tool). q: 5 arm joint angles (rad)."""
    # Design:
    # - Start from the arm base pose (translation BOARD["arm"]["base_xyz"] unless given).
    # - For each joint: T = T @ origin; if it moves, T = T @ Rot(axis, q_i) (formula in the
    #   markdown). Fixed joints consume no q.
    # - Keep every intermediate frame: useful for drawing the arm and for collision checks
    #   along the links (safety TODO), not only the tool.
    # TODO: implement
    raise NotImplementedError

def fk(q) -> np.ndarray:
    """4x4 pose of the tool frame (gripper tip; z axis = approach direction)."""
    # Design: last frame of fk_all.
    # TODO: implement
    raise NotImplementedError

def tool_position(q) -> np.ndarray:
    """(3,) tool position in m."""
    # TODO: implement (translation column of fk)
    raise NotImplementedError

def approach_vector(q) -> np.ndarray:
    """(3,) unit approach direction (tool z axis)."""
    # TODO: implement (third rotation column of fk)
    raise NotImplementedError

def tilt_from_vertical_deg(q) -> float:
    """Angle between the approach vector and straight down (0 = perfectly top-down)."""
    # Design: angle between approach and (0, 0, -1) = acos(-a_z); clip before acos (rounding).
    # TODO: implement
    raise NotImplementedError

PARK = np.array(BOARD["arm"]["park"])
# Try once FK works: tool_position(np.zeros(5)) should be about (0.3914, 0.0, 0.2265);
# tool_position(PARK) should be a safe point above and behind the board.

## Numeric Jacobian

Central differences of the 6-vector task `[position; approach]` with respect to q. Cheap
enough for 5 joints and avoids deriving the analytic Jacobian (do that later in C++ if speed
matters).

In [ ]:
def task_vector(q) -> np.ndarray:
    """6-vector [tool position; approach vector] that IK drives to its target."""
    # TODO: implement
    raise NotImplementedError

def numeric_jacobian(q, eps: float = 1e-6) -> np.ndarray:
    """(6, n) Jacobian of task_vector by central differences."""
    # Design: perturb one joint at a time by +-eps, column j = difference / (2 eps).
    # Central (not forward) differences: error O(eps^2) for 2x the FK calls; 5 joints -> 10 FKs.
    # Decision for you: eps; too small amplifies float rounding, too big smooths curvature.
    # TODO: implement
    raise NotImplementedError

## Inverse kinematics (damped least squares)

Goal: tip at `p*` with approach direction `a*`. Error `e = [p* - p(q); w (a* - a(q))]`
(the weight `w` trades millimeters for radians). DLS step:

$$\Delta q = J^T (J J^T + \lambda^2 I)^{-1} e$$

`lambda` keeps steps bounded near singularities (arm fully stretched or folded). After each
step we clip to the joint limits; that is the crude part (TODO above).

With 5 joints and 5 meaningful constraints (3 position + 2 for a direction), wrist_roll is
nearly free: it only spins the fingers about the approach axis. A good default target is
`approach_from_pitch(p, pitch)`: tilt toward the target from straight down, in the vertical
plane through the arm base.

In [ ]:
def approach_from_pitch(target, pitch_rad: float) -> np.ndarray:
    """Unit approach vector pointing from above toward `target`: pitch = angle below horizontal
    (pi/2 = straight down), leaning away from the arm base in the vertical plane through it."""
    # Design: horizontal unit vector u from the base toward target (x, y); approach =
    # (cos(pitch) u, -sin(pitch)). Guard against target directly above the base (|xy| = 0).
    # TODO: implement
    raise NotImplementedError

def ik_dls(target, approach=None, q0=None, pitch=None, iters=300, lam=0.01, w_orient=0.05,
           pos_tol=1e-4, orient_tol=2e-3, limits=True, lower=None, upper=None):
    """Damped-least-squares IK.
    target: (3,) tip position in m; approach: (3,) unit vector or None (position only).
    pitch: instead of a fixed approach, ask for this angle below horizontal (pi/2 = straight
    down) and let the horizontal direction follow the arm's own vertical plane (re-evaluated
    every iteration). pitch > pi/2 leans the approach back toward the arm. With only 4 joints that move the approach in that plane, a fixed
    approach that is not in the plane is unreachable, so prefer `pitch` for top-down grasps.
    q0: seed (defaults to a bent pose above the board). lower/upper: joint limits to clip to
    (default: the URDF limits; safety.ipynb passes its tighter ones). Returns (q, info) with
    info = {converged, pos_err, orient_err, iters}."""
    # Design:
    # - Iterate: error e = [p* - p(q); a* - a(q)] (position only if no approach), weighted
    #   by W = diag(1,1,1,w,w,w) so meters and unit-vector components are comparable.
    # - Step: dq = J^T (J J^T + lambda^2 I)^-1 e (np.linalg.solve, not inv). lambda bounds the
    #   step near singularities (arm stretched / folded) at the cost of slower convergence.
    # - With `pitch`: recompute a* each iteration from the current pan angle (q[0]): the arm's
    #   vertical plane is x-z rotated by pan, and +pan swings toward -y (see FK notes).
    # - Clip to [lower, upper] after each step (crude; see TODO list for better options).
    # - Stop when both errors are below tolerance; always return the info dict so callers can
    #   tell "converged" from "best effort".
    # Decisions for you: default seed (elbow-up bent pose above the board), lambda, w_orient.
    # TODO: implement
    raise NotImplementedError

def ik_top_down(target, tilt_deg: float = 0.0, q0=None, **kw):
    """IK for a (nearly) top-down approach at `target`.
    Positive tilt leans the approach away from the arm base, negative toward it.
    TODO (pick_place.plan_joint_trajectory will try a list of tilts): choose tilt automatically
    per square within BOARD['arm']['max_tilt_deg'] (try 0, then increase until IK converges
    inside limits), constrain wrist_roll so the fingers close along a file or rank, and return
    None (not a bad q) when no solution exists."""
    # Design: reject |tilt| > arm.max_tilt_deg (ValueError), then ik_dls with
    # pitch = 90 deg - tilt (in radians), passing through q0 and the other keyword args.
    # TODO: implement
    raise NotImplementedError

### Reachability over the board (quick look)

For each square: does top-down IK converge at hover height? (Prints a grid; `.` = converged,
`x` = not.) This is the starting point for the per-square tilt TODO.

In [ ]:
def reachability_grid(height=None, tilt_deg: float = 0.0) -> str:
    """Text grid, rank 8 at the top: '.' = top-down IK converged at hover, 'x' = not."""
    # Design: loop ranks 8..1 and files a..h, ik_top_down at hover_point (adjusted to `height`),
    # mark convergence; add a file legend row. Slow-ish (64 IK solves), so not run on import.
    # TODO: implement
    raise NotImplementedError

# print(reachability_grid())          # ~2 s; uncomment to look once IK works

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - parse_chain(): joint names == ARM_JOINTS + ["gripper_frame_joint"], and xyz/rpy/limits
#   match HARDCODED_CHAIN (catches URDF edits that the copy missed); CHAIN_SOURCE is the URDF
# - fk(zeros): finite, rotation part orthonormal (R R^T = I), tip at (0.3914, 0.0, 0.2265)
#   +- 0.5 mm (same numbers as gambit_cpp/tests/test_kinematics.cpp)
# - sign convention: tool_position([0.5, 0, 0, 0, 0]) has y < 0 (+pan swings toward -y)
# - numeric_jacobian at a random pose: shape (6, 5), finite; compare one column with a
#   hand-made finite difference
# - IK round trip: q_true -> FK -> ik_dls(position, approach, seed q_true + 0.1) converges and
#   the tip is within 1 mm of FK(q_true)
# - ik_top_down(hover_point("e4")) converges with tilt_from_vertical_deg < 1
# Example shape of a test:
# T0 = fk(np.zeros(5)); assert np.allclose(T0[:3, 3], [0.3914, 0.0, 0.2265], atol=5e-4)
print("Self-check: not written yet (see plan above)")
print("kinematics OK")